# Data File Exploration

Files inspected:
   - data/df_sample_small.parquet       (Run 1/2a working sample)
   - data/sample_data3.parquet          (unknown - inspect first)
   - data/masked_transaction_dataset.parquet
   - data/id_mapping_SECURE.parquet     (inspect shape only - do not expose IDs)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

DATA_DIR = Path("data")

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 80)

SPEND_COLOR    = "#3b82f6"
NONSPEND_COLOR = "#f59e0b"
REMOVE_COLOR   = "#ef4444"
KEEP_COLOR     = "#22c55e"

In [ ]:
df_small   = pd.read_parquet(DATA_DIR / "df_sample_small.parquet")
df_sample  = pd.read_parquet(DATA_DIR / "sample_data3.parquet")
df_masked  = pd.read_parquet(DATA_DIR / "masked_transaction_dataset.parquet")
df_id      = pd.read_parquet(DATA_DIR / "id_mapping_SECURE.parquet")

for name, df in [("df_small", df_small), ("df_sample", df_sample),
                 ("df_masked", df_masked), ("df_id", df_id)]:
    print(f"  {name:15s} {df.shape[0]:>8,} rows  {df.shape[1]:>3} cols")


In [ ]:
# ── 2. File overview - shape, columns, overlap ───────────────────────────────
print("\n\n=== SCHEMAS ===\n")
for name, df in [("df_small", df_small), ("df_sample", df_sample), ("df_masked", df_masked)]:
    print(f"--- {name} ---")
    info = pd.DataFrame({
        "dtype":    df.dtypes,
        "nulls_%":  (df.isna().mean() * 100).round(1),
        "sample":   [df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns]
    })
    print(info.to_string())
    print()

# id file - shape only
print(f"--- df_id (SECURE - shape only) ---")
print(f"  {df_id.shape[0]:,} rows  |  columns: {list(df_id.columns)}\n")

# Column overlap
col_sets = {"df_small": set(df_small.columns), "df_sample": set(df_sample.columns),
            "df_masked": set(df_masked.columns)}
all_cols = sorted(set().union(*col_sets.values()))
overlap  = pd.DataFrame({n: [c in s for c in all_cols] for n, s in col_sets.items()}, index=all_cols)
print("=== COLUMN OVERLAP ===")
print(overlap.to_string())

# Transaction ID overlap
if all("transaction_id" in df.columns for df in [df_small, df_sample, df_masked]):
    ids_small  = set(df_small["transaction_id"])
    ids_sample = set(df_sample["transaction_id"])
    ids_masked = set(df_masked["transaction_id"])
    print(f"\ntransaction_id overlaps:")
    print(f"  df_small  ∩ df_sample : {len(ids_small  & ids_sample):,}")
    print(f"  df_small  ∩ df_masked : {len(ids_small  & ids_masked):,}")
    print(f"  df_sample ∩ df_masked : {len(ids_sample & ids_masked):,}")
    


In [ ]:
# ── 3. Dataset cleanup impact ─────────────────────────────────────────────────
# The 3 categories we agreed to remove before Run 2b
gt_col = "base_category_key" if "base_category_key" in df_sample.columns else "ground_truth"

mask_savings  = df_small[gt_col] == "SAVINGS"
mask_uncat    = df_small[gt_col] == "UNCATEGORISED"
mask_roundup  = (
    (df_small[gt_col] == "INTERNAL_TRANSFER") &
    df_small["original_description"].str.contains(r"(?i)round.?up", na=False)
)
mask_remove   = mask_savings | mask_uncat | mask_roundup

n_total    = len(df_small)
n_savings  = mask_savings.sum()
n_uncat    = mask_uncat.sum()
n_roundup  = mask_roundup.sum()
n_remove   = mask_remove.sum()
n_keep     = n_total - n_remove

print(f"\n\n=== CLEANUP IMPACT ===")
print(f"  SAVINGS GT rows     : {n_savings:,}")
print(f"  UNCATEGORISED rows  : {n_uncat:,}")
print(f"  ROUND_UP mislabels  : {n_roundup:,}")
print(f"  Total to remove     : {n_remove:,}  ({n_remove/n_total*100:.1f}%)")
print(f"  Remaining           : {n_keep:,}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Dataset Cleanup Impact", fontweight="bold", fontsize=13)

# Left - stacked bar
categories  = ["SAVINGS", "UNCATEGORISED", "ROUND_UP\nmislabels", "Keep"]
counts      = [n_savings, n_uncat, n_roundup, n_keep]
colors      = [REMOVE_COLOR, REMOVE_COLOR, REMOVE_COLOR, KEEP_COLOR]
bars = axes[0].barh(categories, counts, color=colors, height=0.5)
for bar, count in zip(bars, counts):
    axes[0].text(bar.get_width() + 10, bar.get_y() + bar.get_height()/2,
                 f"{count:,}  ({count/n_total*100:.1f}%)", va="center", fontsize=9)
axes[0].set_xlim(0, n_total * 1.3)
axes[0].set_title("Rows removed vs kept")
axes[0].invert_yaxis()

# Right - GT distribution of remaining rows, highlighting problem categories
gt_counts = df_small[gt_col].value_counts().head(25)
bar_colors = [REMOVE_COLOR if k in ("SAVINGS", "UNCATEGORISED") else SPEND_COLOR
              for k in gt_counts.index]
axes[1].barh(gt_counts.index[::-1], gt_counts.values[::-1], color=bar_colors[::-1], height=0.7)
axes[1].set_title("GT distribution - top 25\n(red = removed categories)")
axes[1].set_xlabel("Count")
remove_patch = mpatches.Patch(color=REMOVE_COLOR, label="Removed")
keep_patch   = mpatches.Patch(color=SPEND_COLOR,  label="Kept")
axes[1].legend(handles=[remove_patch, keep_patch], fontsize=8)

plt.tight_layout()
plt.savefig("outputs/nb6_cleanup_impact.png", dpi=150, bbox_inches="tight")
plt.show()



In [ ]:

# ── 4. Merchant signal coverage ───────────────────────────────────────────────
# Key question: how many transactions have a usable merchant signal?
# merchant_name populated = strong spend signal; absent = lean non-spend

print("\n\n=== MERCHANT SIGNAL COVERAGE ===")

df_clean = df_small[~mask_remove].copy()

merchant_present = df_clean["merchant_name"].notna() & (df_clean["merchant_name"].str.strip() != "")

print(f"Overall merchant_name populated: {merchant_present.sum():,} / {len(df_clean):,} "
      f"({merchant_present.mean()*100:.1f}%)")

# By provider
prov_merchant = (
    df_clean.groupby("provider_name")["merchant_name"]
    .apply(lambda x: x.notna().mean() * 100)
    .sort_values(ascending=True)
    .round(1)
)

# By spend_type (using taxonomy_master_updated if available, else GT heuristic)
# Fallback: positive amount = likely spend
df_clean["merchant_present"] = merchant_present

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Merchant Signal Coverage (cleaned sample)", fontweight="bold", fontsize=13)

# Left - by provider
colors_prov = [KEEP_COLOR if v >= 50 else REMOVE_COLOR for v in prov_merchant.values]
axes[0].barh(prov_merchant.index, prov_merchant.values, color=colors_prov, height=0.7)
axes[0].axvline(50, color="black", linestyle="--", linewidth=1, label="50% threshold")
for i, v in enumerate(prov_merchant.values):
    axes[0].text(v + 0.5, i, f"{v:.0f}%", va="center", fontsize=8)
axes[0].set_xlim(0, 115)
axes[0].set_title("merchant_name populated % by provider")
axes[0].set_xlabel("% transactions with merchant_name")
axes[0].legend(fontsize=8)

# Right - merchant present vs absent, split by GT spend type (rough proxy via amount)
df_clean["amount_sign"] = df_clean["amount"].apply(lambda x: "credit (>0)" if x > 0 else "debit (<0)")
merchant_by_sign = df_clean.groupby("amount_sign")["merchant_present"].mean() * 100

axes[1].bar(merchant_by_sign.index, merchant_by_sign.values,
            color=[NONSPEND_COLOR, SPEND_COLOR], width=0.4)
for i, (k, v) in enumerate(merchant_by_sign.items()):
    axes[1].text(i, v + 1, f"{v:.1f}%", ha="center", fontsize=10, fontweight="bold")
axes[1].set_ylim(0, 110)
axes[1].set_title("Merchant present % by transaction direction\n(credit = likely non-spend, debit = likely spend)")
axes[1].set_ylabel("% with merchant_name populated")

plt.tight_layout()
plt.savefig("outputs/nb6_merchant_coverage.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:

# ── 5. Spend vs non-spend balance ─────────────────────────────────────────────
# Understand current class balance and what changes after cleanup

print("\n\n=== SPEND / NON-SPEND BALANCE ===")

# Load taxonomy_master_updated to get spend_type per key
try:
    tax = pd.read_csv("outputs/taxonomy_master_updated.csv.txt")
    leans_map = tax.set_index("base_category_key")["leans"].to_dict()
    df_clean["spend_type_gt"] = df_clean[gt_col].map(leans_map)
    print("spend_type resolved via taxonomy_master_updated")
except Exception:
    # Fallback: amount-based heuristic
    df_clean["spend_type_gt"] = df_clean["amount"].apply(
        lambda x: "non_spend" if x > 0 else "spend"
    )
    print("spend_type estimated from amount sign (taxonomy file not found)")

balance = df_clean["spend_type_gt"].value_counts()
print(balance.to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Spend vs Non-spend Balance (cleaned sample)", fontweight="bold", fontsize=13)

# Left - overall pie
axes[0].pie(balance.values, labels=balance.index, autopct="%1.1f%%",
            colors=[SPEND_COLOR, NONSPEND_COLOR], startangle=90)
axes[0].set_title("Overall balance")

# Right - by provider
prov_balance = (
    df_clean[df_clean["spend_type_gt"].notna()]
    .groupby(["provider_name", "spend_type_gt"])
    .size()
    .unstack(fill_value=0)
)
prov_balance_pct = prov_balance.div(prov_balance.sum(axis=1), axis=0) * 100
prov_balance_pct = prov_balance_pct.sort_values(
    "spend" if "spend" in prov_balance_pct.columns else prov_balance_pct.columns[0]
)
prov_balance_pct.plot(kind="barh", stacked=True, ax=axes[1],
                      color=[SPEND_COLOR, NONSPEND_COLOR], width=0.7)
axes[1].set_title("Spend / non-spend split by provider")
axes[1].set_xlabel("% of transactions")
axes[1].legend(loc="lower right", fontsize=8)

plt.tight_layout()
plt.savefig("outputs/nb6_spend_balance.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:

# ── 6. df_sample - flag X3 post-processed transactions ───────────────────────
# PO confirmed: rows where properties contains 'ml_transfer_sub_category_key'
# have been post-processed by X3 and should be excluded from the sample.

print("\n\n=== df_sample - X3 FLAG ===")
print(f"Shape: {df_sample.shape}")
print(f"\nproperties sample values (3):")
print(df_sample["properties"].dropna().head(3).to_string())

# Flag rows touched by X3
df_sample["x3_processed"] = df_sample["properties"].str.contains(
    "ml_transfer_sub_category_key", case=False, na=False
)

print(f"\nx3_processed distribution:")
print(df_sample["x3_processed"].value_counts())
print(f"\n  {(~df_sample['x3_processed']).sum():,} rows remain after excluding X3")

In [ ]:
# ── X3 breakdown by provider and spend/non-spend ──────────────────────────────

# Resolve spend_type for df_sample
try:
    tax       = pd.read_csv("outputs/taxonomy_master_updated.csv.txt")
    leans_map = tax.set_index("base_category_key")["leans"].to_dict()
    df_sample["spend_type_gt"] = df_sample["base_category_key"].map(leans_map)
except Exception:
    df_sample["spend_type_gt"] = df_sample["amount"].apply(
        lambda x: "non_spend" if x > 0 else "spend"
    )

# ── Summary stats ──────────────────────────────────────────────────────────────
total     = len(df_sample)
n_x3      = df_sample["x3_processed"].sum()
n_not_x3  = total - n_x3

print(f"Total rows      : {total:,}")
print(f"X3 processed    : {n_x3:,}  ({n_x3/total*100:.1f}%)")
print(f"Not X3          : {n_not_x3:,}  ({n_not_x3/total*100:.1f}%)")

# ── By provider ───────────────────────────────────────────────────────────────
prov_x3 = (
    df_sample.groupby("provider_name")["x3_processed"]
    .agg(["sum", "count"])
    .rename(columns={"sum": "x3", "count": "total"})
    .assign(x3_pct=lambda d: (d["x3"] / d["total"] * 100).round(1),
            not_x3=lambda d: d["total"] - d["x3"])
    .sort_values("x3_pct", ascending=True)
)

# ── By spend type ─────────────────────────────────────────────────────────────
spend_x3 = (
    df_sample.groupby("spend_type_gt")["x3_processed"]
    .agg(["sum", "count"])
    .rename(columns={"sum": "x3", "count": "total"})
    .assign(x3_pct=lambda d: (d["x3"] / d["total"] * 100).round(1))
)

print(f"\nBy spend type:\n{spend_x3.to_string()}")

# ── Visuals ───────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("X3 Post-processed Transactions", fontweight="bold", fontsize=13)

# Left - stacked bar by provider
axes[0].barh(prov_x3.index, prov_x3["not_x3"], color=KEEP_COLOR,   height=0.7, label="Not X3")
axes[0].barh(prov_x3.index, prov_x3["x3"],     color=REMOVE_COLOR, height=0.7,
             left=prov_x3["not_x3"], label="X3 processed")
for i, (_, row) in enumerate(prov_x3.iterrows()):
    axes[0].text(row["total"] + 10, i, f"{row['x3_pct']:.0f}%", va="center", fontsize=8)
axes[0].set_title("X3 processed vs not - by provider")
axes[0].set_xlabel("Transaction count")
axes[0].legend(fontsize=8)

# Right - stacked bar by spend type
spend_not_x3 = spend_x3["total"] - spend_x3["x3"]
x = range(len(spend_x3))
axes[1].bar(x, spend_not_x3,       color=KEEP_COLOR,   width=0.4, label="Not X3")
axes[1].bar(x, spend_x3["x3"],     color=REMOVE_COLOR, width=0.4,
            bottom=spend_not_x3, label="X3 processed")
axes[1].set_xticks(x)
axes[1].set_xticklabels(spend_x3.index)
for i, (_, row) in enumerate(spend_x3.iterrows()):
    axes[1].text(i, row["total"] + 20, f"{row['x3_pct']:.0f}%", ha="center", fontsize=10,
                 fontweight="bold")
axes[1].set_title("X3 processed vs not - by spend type")
axes[1].set_ylabel("Transaction count")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig("outputs/nb6_x3_breakdown.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── X3 breakdown by base_category_key ────────────────────────────────────────

cat_x3 = (
    df_sample.groupby("base_category_key")["x3_processed"]
    .agg(["sum", "count"])
    .rename(columns={"sum": "x3", "count": "total"})
    .assign(x3_pct=lambda d: (d["x3"] / d["total"] * 100).round(1))
    .sort_values("x3", ascending=False)
)

print(cat_x3.to_string())

# ── Two charts: volume and rate ───────────────────────────────────────────────
# Focus on categories where X3 actually appears
cat_plot = cat_x3[cat_x3["x3"] > 0].sort_values("x3", ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(14, max(6, len(cat_plot) * 0.35)))
fig.suptitle("X3 Post-processed - by base_category_key", fontweight="bold", fontsize=13)

# Left - raw X3 count per category
axes[0].barh(cat_plot.index, cat_plot["x3"], color=REMOVE_COLOR, height=0.7)
axes[0].set_title("X3 count by category")
axes[0].set_xlabel("X3 processed rows")

# Right - X3 rate (%) per category
colors = [REMOVE_COLOR if v == 100 else NONSPEND_COLOR if v >= 50 else SPEND_COLOR
          for v in cat_plot["x3_pct"]]
axes[1].barh(cat_plot.index, cat_plot["x3_pct"], color=colors, height=0.7)
axes[1].axvline(50, color="black", linestyle="--", linewidth=1)
for i, v in enumerate(cat_plot["x3_pct"]):
    axes[1].text(v + 0.5, i, f"{v:.0f}%", va="center", fontsize=8)
axes[1].set_xlim(0, 115)
axes[1].set_title("X3 rate (%) by category\n(red = 100%, orange = >50%)")
axes[1].set_xlabel("% of category rows that are X3 processed")

plt.tight_layout()
plt.savefig("outputs/nb6_x3_by_category.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Check which base_category_keys in df_sample are not in the taxonomy ───────

valid_keys = set(pd.read_csv("assets/cat_keys.csv")["base_category_key"].str.strip().str.upper())

all_keys  = set(df_sample["base_category_key"].str.strip().str.upper().dropna().unique())
invalid   = all_keys - valid_keys

print(f"Unique keys in df_sample : {len(all_keys)}")
print(f"Valid taxonomy keys      : {len(valid_keys)}")
print(f"Not in taxonomy          : {len(invalid)}")

if invalid:
    # Show count and X3 rate for each invalid key
    inv_df = (
        df_sample[df_sample["base_category_key"].str.upper().isin(invalid)]
        .groupby("base_category_key")["x3_processed"]
        .agg(total="count", x3="sum")
        .assign(x3_pct=lambda d: (d["x3"] / d["total"] * 100).round(1))
        .sort_values("total", ascending=False)
    )
    print(f"\nInvalid keys — row counts and X3 rate:")
    print(inv_df.to_string())

In [ ]:
!find . -name "*cat_keys*" -o -name "*cat_key*"

In [ ]:
pd.read_csv("assets/cat_keys.csv").head(10)

In [ ]:
# Step 1 - remove X3 post-processed rows
df_sample["x3_processed"] = df_sample["properties"].str.contains(
    "ml_transfer_sub_category_key", case=False, na=False
)
df_work = df_sample[~df_sample["x3_processed"]].copy()

# Step 2 - remove GT cleanup categories
mask_remove = (
    (df_work["base_category_key"] == "SAVINGS") |
    (df_work["base_category_key"] == "UNCATEGORISED") |
    (
        (df_work["base_category_key"] == "INTERNAL_TRANSFER") &
        df_work["original_description"].str.contains(r"(?i)round.?up", na=False)
    )
)
df_clean = df_work[~mask_remove].copy()

In [ ]:
df_clean.shape

In [ ]:
#11: Export
OUTPUT_PATH = './data/clean_data1.parquet'

df_clean.to_parquet(OUTPUT_PATH, index=False)
print(f"Dataset saved: {OUTPUT_PATH} ({len(df_masked):,} rows)")